In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir(r"C:\Users\A_PORTEU\python_projects\lib_climate_alignment")

In [6]:
first_year_available, last_year_available = 2019, 2024

df = pd.read_parquet(f'data/intermediate_data/df_merged_all_infos_{first_year_available}_{last_year_available}.parquet')
df_hist_abs_emissions = pd.read_parquet(f'data/intermediate_data/hist_abs_emissions_{first_year_available}_{last_year_available}.parquet')
df_abs_em_rate = pd.read_parquet(f'data/intermediate_data/hist_abs_emissions_growth_rate_{first_year_available}_{last_year_available}.parquet')
df_hist_intensities = pd.read_parquet(f'data/intermediate_data/hist_intensities_{first_year_available}_{last_year_available}.parquet')
df_intensities_rate = pd.read_parquet(f'data/intermediate_data/hist_intensities_growth_rate_{first_year_available}_{last_year_available}.parquet')
df_auto_intensities_rate = pd.read_parquet(f"data/intermediate_data/df_auto_intensities_rate_{first_year_available}_{last_year_available}.parquet")

scopes = ["s1", "s2", "s3_d", "s3_u"]
regions = list(df['region_0'].unique())
list_high_impact_sector = list(df['high_impact_sector'].dropna().sort_values().unique())
years_str = sorted([c for c in df_hist_abs_emissions.columns if c.isdigit()], key=int, reverse=False)
dict_year_delta_years = {y: f"{int(y)-1}-{y}" for y in years_str[1:]}
delta_years_str = list(dict_year_delta_years.values())

df_eq_abs_em = pd.merge(df[['isin', 'scope','high_impact_sector', 'region_0', 'nace']], df_hist_abs_emissions, how='left', on=['isin', 'scope'])
df_eq_abs_em_rate = pd.merge(df[['isin', 'scope','high_impact_sector', 'region_0']], df_abs_em_rate, how='left', on=['isin', 'scope'])
df_eq_intensities = pd.merge(df[['isin', 'scope','high_impact_sector', 'region_0']], df_hist_intensities, how='left', on=['isin', 'scope'])
df_eq_intensities_rate = pd.merge(df[['isin', 'scope','high_impact_sector', 'region_0']], 
                             df_intensities_rate[['isin', 'scope'] + delta_years_str], how='left', 
                             on=['isin', 'scope'])
df_eq_auto_intensities_rate = pd.merge(df[['isin', 'scope','high_impact_sector', 'region_0']], 
                             df_auto_intensities_rate[['isin', 'scope'] + delta_years_str], how='left', 
                             on=['isin', 'scope'])



# Automobiles

In [7]:
list_eq_auto = list(df[df['high_impact_sector']=="Automobiles"]['isin'].unique())

## Les outliers des emissions absolues s3_d

In [8]:
df_abs_em_rate_auto = df_abs_em_rate[(df_abs_em_rate['isin'].isin(list_eq_auto))&(df_abs_em_rate['scope']=="s3_d")].copy()
df_abs_em_rate_auto['scope'] = df_abs_em_rate_auto['scope'].astype(str) 

In [9]:
# Stat. des. Number of observations
df_abs_em_rate_auto['nbr_observations'] = df_abs_em_rate_auto[delta_years_str].notna().sum(axis=1)
# df_abs_em_rate_auto.groupby('scope')['nbr_observations'].describe()

In [10]:
# Stat. des. Average of the rates
df_abs_em_rate_auto['average_rate'] = df_abs_em_rate_auto[delta_years_str].mean(axis=1)
# df_abs_em_rate_auto.groupby('scope')['average_rate'].describe()

In [11]:
# Stat. des. Number observations inferior to -0.2% 
# Stat. des. Number observations superior to 0.2%
df_abs_em_rate_auto = df_abs_em_rate_auto.copy()
df_abs_em_rate_auto['number_sup_10'] = (df_abs_em_rate_auto[delta_years_str] > 0.1).sum(axis=1)
df_abs_em_rate_auto['number_inf_10'] = (df_abs_em_rate_auto[delta_years_str] < -0.1).sum(axis=1)
df_abs_em_rate_auto['number_sup_20'] = (df_abs_em_rate_auto[delta_years_str] > 0.2).sum(axis=1)
df_abs_em_rate_auto['number_inf_20'] = (df_abs_em_rate_auto[delta_years_str] < -0.2).sum(axis=1)
df_abs_em_rate_auto['number_sup_or_inf_20'] = ((df_abs_em_rate_auto[delta_years_str] > 0.2)|(df_abs_em_rate_auto[delta_years_str] < -0.2)).sum(axis=1)

# df_abs_em_rate_auto.groupby('scope')['number_inf_20'].describe()
# df_abs_em_rate_auto.groupby('scope')['number_sup_20'].describe()
# df_abs_em_rate_auto.groupby('scope')['number_sup_or_inf_20'].describe()


In [12]:
pd.DataFrame({
    "inf to -20%": [(df_abs_em_rate_auto["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_abs_em_rate_auto["number_sup_20"] > 0).sum()],
    "both": [(df_abs_em_rate_auto["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,114,158,167


In [13]:
# Number of observations / Number of rate sup to 20%
# pd.crosstab(
#     [df_abs_em_rate_auto["scope"], df_abs_em_rate_auto["nbr_observations"]],
#     df_abs_em_rate_auto["number_inf_20"]
# )

In [14]:
# Number of observations / Number of rate inf to -20%
# pd.crosstab(
#     [df_abs_em_rate_auto["scope"], df_abs_em_rate_auto["nbr_observations"]],
#     df_abs_em_rate_auto["number_sup_or_inf_20"]
# )

In [15]:
# 1 - Les outliers sont ceux qui ont plus d'un outlier sur au moins 3 observations et un outlier si moins d'observations

In [16]:
df_abs_em_rate_auto['test_1_keep'] = df_abs_em_rate_auto.apply(
    lambda x: True if ((x['nbr_observations']==4 and (x['number_sup_or_inf_20']<2)) or (
        x['nbr_observations']==3 and (x['number_sup_or_inf_20']<2)) or (
        x['nbr_observations'] in [1,2] and (x['number_sup_or_inf_20']==0))) else False, 
        axis=1)
# pd.concat([df_abs_em_rate_auto.groupby('scope')['test_1_keep'].sum(), df_abs_em_rate_auto.groupby("scope")["test_1_keep"].apply(lambda x: (~x).sum())], axis=1)

In [17]:
list_outliers_test_1 = df_abs_em_rate_auto[~df_abs_em_rate_auto["test_1_keep"]]['isin'].to_list()

### apply the 1st adjustment

In [18]:
df_abs_em_rate_auto_ajd_1 = df_abs_em_rate_auto.copy()
df_abs_em_rate_auto_ajd_1.loc[df_abs_em_rate_auto_ajd_1['test_1_keep'], delta_years_str] = (
    df_abs_em_rate_auto_ajd_1.loc[df_abs_em_rate_auto_ajd_1['test_1_keep'], delta_years_str]
    .mask((df_abs_em_rate_auto_ajd_1[delta_years_str] > 0.2) | (df_abs_em_rate_auto_ajd_1[delta_years_str] < -0.2))
)

In [19]:
# Stat. des. Average of the rates
df_abs_em_rate_auto_ajd_1['average_rate'] = df_abs_em_rate_auto_ajd_1[delta_years_str].mean(axis=1)
# df_abs_em_rate_auto_ajd_1.groupby('scope')['average_rate'].describe()

In [20]:
df_abs_em_rate_auto_ajd_1 = df_abs_em_rate_auto_ajd_1.copy()
df_abs_em_rate_auto_ajd_1['number_sup_20'] = (df_abs_em_rate_auto_ajd_1[delta_years_str] > 0.2).sum(axis=1)
df_abs_em_rate_auto_ajd_1['number_inf_20'] = (df_abs_em_rate_auto_ajd_1[delta_years_str] < -0.2).sum(axis=1)
df_abs_em_rate_auto_ajd_1['number_sup_or_inf_20'] = ((df_abs_em_rate_auto_ajd_1[delta_years_str] > 0.2)|(df_abs_em_rate_auto_ajd_1[delta_years_str] < -0.2)).sum(axis=1)
pd.DataFrame({
    "inf to -20%": [(df_abs_em_rate_auto_ajd_1["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_abs_em_rate_auto_ajd_1["number_sup_20"] > 0).sum()],
    "both": [(df_abs_em_rate_auto_ajd_1["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,113,155,163


## Les outliers des intensites (revenues)

In [21]:
df_intensities_rate_auto = df_intensities_rate[(df_intensities_rate['isin'].isin(list_outliers_test_1))&(df_intensities_rate['scope']=="s3_d")].copy()
df_intensities_rate_auto['scope'] = df_intensities_rate_auto['scope'].astype(str) 

In [22]:
# Stat. des. Number of observations
df_intensities_rate_auto['nbr_observations'] = df_intensities_rate_auto[delta_years_str].notna().sum(axis=1)
# df_intensities_rate_auto.groupby('scope')['nbr_observations'].describe()

In [23]:
# Stat. des. Average of the rates
df_intensities_rate_auto['average_rate'] = df_intensities_rate_auto[delta_years_str].mean(axis=1)
# df_intensities_rate_auto.groupby('scope')['average_rate'].describe()

In [24]:
df_intensities_rate_auto = df_intensities_rate_auto.copy()
df_intensities_rate_auto['number_sup_10'] = (df_intensities_rate_auto[delta_years_str] > 0.1).sum(axis=1)
df_intensities_rate_auto['number_inf_10'] = (df_intensities_rate_auto[delta_years_str] < -0.1).sum(axis=1)
df_intensities_rate_auto['number_sup_20'] = (df_intensities_rate_auto[delta_years_str] > 0.2).sum(axis=1)
df_intensities_rate_auto['number_inf_20'] = (df_intensities_rate_auto[delta_years_str] < -0.2).sum(axis=1)
df_intensities_rate_auto['number_sup_or_inf_20'] = ((df_intensities_rate_auto[delta_years_str] > 0.2)|(df_intensities_rate_auto[delta_years_str] < -0.2)).sum(axis=1)

# df_intensities_rate_auto.groupby('scope')['number_inf_20'].describe()
# df_intensities_rate_auto.groupby('scope')['number_sup_20'].describe()
df_intensities_rate_auto.groupby('scope')['number_sup_or_inf_20'].describe()


,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,174.0,2.431034,1.331418,0.0,2.0,2.5,3.0,5.0


In [25]:
# Number of observations / Number of rate sup to 20%
pd.crosstab(
    [df_intensities_rate_auto["scope"], df_intensities_rate_auto["nbr_observations"]],
    df_intensities_rate_auto["number_sup_20"]
)

number_sup_20            0   1   2   3  4
scope nbr_observations                   
s3_d  0                 12   0   0   0  0
      1                  7   2   0   0  0
      2                  1   8   3   0  0
      3                  3  12   2   0  0
      4                  3   5   6   1  0
      5                  1  53  39  12  4

In [26]:
df_intensities_rate_auto['test_2_keep'] = df_intensities_rate_auto.apply(
    lambda x: True if ((x['nbr_observations']==4 and (x['number_sup_or_inf_20']<2)) or (
        x['nbr_observations']==3 and (x['number_sup_or_inf_20']<2)) or (
        x['nbr_observations'] in [1,2] and (x['number_sup_or_inf_20']==0))) else False, 
        axis=1)
# pd.concat([df_intensities_rate_auto.groupby('scope')['test_2_keep'].sum(), df_intensities_rate_auto.groupby("scope")["test_2_keep"].apply(lambda x: (~x).sum())], axis=1)

In [27]:
list_outliers_test_2 = df_intensities_rate_auto[~df_intensities_rate_auto["test_2_keep"]]['isin'].to_list()

In [28]:
pd.DataFrame({
    "inf to -20%": [(df_intensities_rate_auto["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_intensities_rate_auto["number_sup_20"] > 0).sum()],
    "both": [(df_intensities_rate_auto["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,124,147,158


### apply the 2nd adjustment

In [29]:
df_intensities_rate_auto_ajd_2 = df_intensities_rate_auto.copy()
df_intensities_rate_auto_ajd_2.loc[df_intensities_rate_auto_ajd_2['test_2_keep'], delta_years_str] = (
    df_intensities_rate_auto_ajd_2.loc[df_intensities_rate_auto_ajd_2['test_2_keep'], delta_years_str]
    .mask((df_intensities_rate_auto_ajd_2[delta_years_str] > 0.2) | (df_intensities_rate_auto_ajd_2[delta_years_str] < -0.2))
)

In [30]:
# Stat. des. Average of the rates
df_intensities_rate_auto_ajd_2['average_rate'] = df_intensities_rate_auto_ajd_2[delta_years_str].mean(axis=1)
df_intensities_rate_auto_ajd_2.groupby('scope')['average_rate'].describe()

,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,162.0,216.191533,2689.237396,-0.952541,0.267332,0.777217,1.887718,34232.724972


In [31]:
df_intensities_rate_auto_ajd_2 = df_intensities_rate_auto_ajd_2.copy()
df_intensities_rate_auto_ajd_2['number_sup_20'] = (df_intensities_rate_auto_ajd_2[delta_years_str] > 0.2).sum(axis=1)
df_intensities_rate_auto_ajd_2['number_inf_20'] = (df_intensities_rate_auto_ajd_2[delta_years_str] < -0.2).sum(axis=1)
df_intensities_rate_auto_ajd_2['number_sup_or_inf_20'] = ((df_intensities_rate_auto_ajd_2[delta_years_str] > 0.2)|(df_intensities_rate_auto_ajd_2[delta_years_str] < -0.2)).sum(axis=1)
pd.DataFrame({
    "inf to -20%": [(df_intensities_rate_auto_ajd_2["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_intensities_rate_auto_ajd_2["number_sup_20"] > 0).sum()],
    "both": [(df_intensities_rate_auto_ajd_2["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,122,146,155


## Les outliers des physical intensites

In [32]:
len(list_outliers_test_2)

167

In [33]:
df_phys_intensities_rate_auto = df_auto_intensities_rate[(df_auto_intensities_rate['isin'].isin(list_outliers_test_2))&(df_auto_intensities_rate['scope']=="s3_d")].copy()
df_phys_intensities_rate_auto['scope'] = df_phys_intensities_rate_auto['scope'].astype(str) 
df_phys_intensities_rate_auto = pd.concat([df_phys_intensities_rate_auto, pd.DataFrame({
    'isin': [isin for isin in list_outliers_test_2 if isin not in list(df_phys_intensities_rate_auto['isin'])],
    'scope': 's3_d', '2019-2020': np.nan, '2020-2021': np.nan, '2021-2022': np.nan, '2022-2023': np.nan})])#['isin'].nunique()

In [34]:
# Stat. des. Number of observations
df_phys_intensities_rate_auto['nbr_observations'] = df_phys_intensities_rate_auto[delta_years_str].notna().sum(axis=1)
df_phys_intensities_rate_auto.groupby('scope')['nbr_observations'].describe()

,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,167.0,0.269461,0.901385,0.0,0.0,0.0,0.0,4.0


In [35]:
# Stat. des. Average of the rates
df_phys_intensities_rate_auto['average_rate'] = df_phys_intensities_rate_auto[delta_years_str].mean(axis=1)
df_phys_intensities_rate_auto.groupby('scope')['average_rate'].describe()

,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,15.0,-0.026084,0.05049,-0.106984,-0.049794,-0.038978,-0.01456,0.095606


In [36]:
df_phys_intensities_rate_auto = df_phys_intensities_rate_auto.copy()
df_phys_intensities_rate_auto['number_sup_10'] = (df_phys_intensities_rate_auto[delta_years_str] > 0.1).sum(axis=1)
df_phys_intensities_rate_auto['number_inf_10'] = (df_phys_intensities_rate_auto[delta_years_str] < -0.1).sum(axis=1)
df_phys_intensities_rate_auto['number_sup_20'] = (df_phys_intensities_rate_auto[delta_years_str] > 0.2).sum(axis=1)
df_phys_intensities_rate_auto['number_inf_20'] = (df_phys_intensities_rate_auto[delta_years_str] < -0.2).sum(axis=1)
df_phys_intensities_rate_auto['number_sup_or_inf_20'] = ((df_phys_intensities_rate_auto[delta_years_str] > 0.2)|(df_phys_intensities_rate_auto[delta_years_str] < -0.2)).sum(axis=1)

# df_phys_intensities_rate_auto.groupby('scope')['number_inf_20'].describe()
# df_phys_intensities_rate_auto.groupby('scope')['number_sup_20'].describe()
# df_phys_intensities_rate_auto.groupby('scope')['number_sup_or_inf_20'].describe()


In [37]:
# Number of observations / Number of rate sup to 20%
pd.crosstab(
    [df_phys_intensities_rate_auto["scope"], df_phys_intensities_rate_auto["nbr_observations"]],
    df_phys_intensities_rate_auto["number_sup_or_inf_20"]
)

number_sup_or_inf_20      0
scope nbr_observations     
s3_d  0                 152
      1                   1
      2                   3
      3                   6
      4                   5

## Data

In [38]:
list_eq_auto = list(df[df['high_impact_sector']=="Automobiles"]['isin'].unique())

In [39]:
df_abs_em_rate_auto = df_abs_em_rate[(df_abs_em_rate['isin'].isin(list_eq_auto))&(df_abs_em_rate['scope']=="s3_d")].copy()
df_abs_em_rate_auto['scope'] = df_abs_em_rate_auto['scope'].astype(str)
df_intensities_rate_auto = df_intensities_rate[(df_intensities_rate['isin'].isin(list_eq_auto))&(df_intensities_rate['scope']=="s3_d")].copy()
df_intensities_rate_auto['scope'] = df_intensities_rate_auto['scope'].astype(str)
df_auto_intensities_rate = df_auto_intensities_rate[(df_auto_intensities_rate['isin'].isin(list_eq_auto))&(df_auto_intensities_rate['scope']=="s3_d")].copy()
df_auto_intensities_rate['scope'] = df_auto_intensities_rate['scope'].astype(str)


## Before adjustment

In [40]:
# Stat. des. Average of the rates
df_abs_em_rate_auto['average_rate'] = df_abs_em_rate_auto[delta_years_str].mean(axis=1)
df_abs_em_rate_auto.groupby('scope')['average_rate'].describe()

,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,172.0,232.396788,2971.561769,-0.794042,0.515008,1.048598,2.147125,38976.849856


In [41]:
# Description outliers
df_abs_em_rate_auto['number_sup_20'] = (df_abs_em_rate_auto[delta_years_str] > 0.2).sum(axis=1)
df_abs_em_rate_auto['number_inf_20'] = (df_abs_em_rate_auto[delta_years_str] < -0.2).sum(axis=1)
df_abs_em_rate_auto['number_sup_or_inf_20'] = ((df_abs_em_rate_auto[delta_years_str] > 0.2)|(df_abs_em_rate_auto[delta_years_str] < -0.2)).sum(axis=1)
pd.DataFrame({
    "inf to -20%": [(df_abs_em_rate_auto["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_abs_em_rate_auto["number_sup_20"] > 0).sum()],
    "both": [(df_abs_em_rate_auto["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,114,158,167


## Adjustment 1

In [42]:
df_abs_em_rate_auto['number_sup_20'] = (df_abs_em_rate_auto[delta_years_str] > 0.2).sum(axis=1)
df_abs_em_rate_auto['number_inf_20'] = (df_abs_em_rate_auto[delta_years_str] < -0.2).sum(axis=1)
df_abs_em_rate_auto['number_sup_or_inf_20'] = ((df_abs_em_rate_auto[delta_years_str] > 0.2)|(df_abs_em_rate_auto[delta_years_str] < -0.2)).sum(axis=1)
df_abs_em_rate_auto['nbr_observations'] = df_abs_em_rate_auto[delta_years_str].notna().sum(axis=1)
df_abs_em_rate_auto['can_remove_outlier'] = df_abs_em_rate_auto.apply(
    lambda x: True if ((x['nbr_observations']==4 and (x['number_sup_or_inf_20']<2)) or (
        x['nbr_observations']==3 and (x['number_sup_or_inf_20']<2)) or (
        x['nbr_observations'] in [1,2] and (x['number_sup_or_inf_20']==0))) else False, 
        axis=1)
df_ajd_1 = df_abs_em_rate_auto.copy()
df_ajd_1.loc[df_ajd_1['can_remove_outlier'], delta_years_str] = (
    df_ajd_1.loc[df_ajd_1['can_remove_outlier'], delta_years_str]
    .mask((df_ajd_1[delta_years_str] > 0.2) | (df_ajd_1[delta_years_str] < -0.2))
)

In [43]:
# Stat. des. Average of the rates
df_ajd_1['average_rate'] = df_ajd_1[delta_years_str].mean(axis=1)
df_ajd_1.groupby('scope')['average_rate'].describe()

,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,172.0,232.392077,2971.562139,-0.794042,0.469037,1.048598,2.147125,38976.849856


In [44]:
# Description outliers
df_ajd_1['number_sup_20'] = (df_ajd_1[delta_years_str] > 0.2).sum(axis=1)
df_ajd_1['number_inf_20'] = (df_ajd_1[delta_years_str] < -0.2).sum(axis=1)
df_ajd_1['number_sup_or_inf_20'] = ((df_ajd_1[delta_years_str] > 0.2)|(df_ajd_1[delta_years_str] < -0.2)).sum(axis=1)
pd.DataFrame({
    "inf to -20%": [(df_ajd_1["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_ajd_1["number_sup_20"] > 0).sum()],
    "both": [(df_ajd_1["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,113,155,163


## Adjustment 2

In [45]:
df_intensities_rate_auto = df_intensities_rate_auto.copy()
df_intensities_rate_auto['nbr_observations'] = df_intensities_rate_auto[delta_years_str].notna().sum(axis=1)
df_intensities_rate_auto['number_sup_20'] = (df_intensities_rate_auto[delta_years_str] > 0.2).sum(axis=1)
df_intensities_rate_auto['number_inf_20'] = (df_intensities_rate_auto[delta_years_str] < -0.2).sum(axis=1)
df_intensities_rate_auto['number_sup_or_inf_20'] = ((df_intensities_rate_auto[delta_years_str] > 0.2)|(df_intensities_rate_auto[delta_years_str] < -0.2)).sum(axis=1)
# Number of observations / Number of rate sup to 20%
pd.crosstab(
    [df_intensities_rate_auto["scope"], df_intensities_rate_auto["nbr_observations"]],
    df_intensities_rate_auto["number_sup_or_inf_20"]
)

number_sup_or_inf_20     0   1   2   3   4   5
scope nbr_observations                        
s3_d  0                 12   0   0   0   0   0
      1                  4   7   0   0   0   0
      2                  2   4   9   0   0   0
      3                  2   6   6   7   0   0
      4                  1   0   4   6   4   0
      5                  0  14  25  37  23  10

In [46]:
df_intensities_rate_auto = df_intensities_rate_auto.set_index('isin')

In [47]:
df_ajd_2 = df_ajd_1.copy()
df_ajd_2 = df_ajd_2.set_index('isin')
df_ajd_2[delta_years_str] = (df_ajd_2[delta_years_str]
    .mask(((df_ajd_2[delta_years_str]>0.2)|(df_ajd_2[delta_years_str]<-0.2))&
          (df_intensities_rate_auto[delta_years_str].notna()), 
    df_intensities_rate_auto[delta_years_str])
)

In [48]:
df_ajd_2['average_rate'] = df_ajd_2[delta_years_str].mean(axis=1)
df_ajd_2.groupby('scope')['average_rate'].describe()

,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,172.0,204.728765,2609.85255,-0.81902,0.271936,0.855975,1.925302,34232.753671


In [49]:
df_ajd_2['number_sup_20'] = (df_ajd_2[delta_years_str] > 0.2).sum(axis=1)
df_ajd_2['number_inf_20'] = (df_ajd_2[delta_years_str] < -0.2).sum(axis=1)
df_ajd_2['number_sup_or_inf_20'] = ((df_ajd_2[delta_years_str] > 0.2)|(df_ajd_2[delta_years_str] < -0.2)).sum(axis=1)
pd.DataFrame({
    "inf to -20%": [(df_ajd_2["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_ajd_2["number_sup_20"] > 0).sum()],
    "both": [(df_ajd_2["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,111,150,161


## Adjustment 3

In [50]:
df_auto_intensities_rate = df_auto_intensities_rate.copy()
df_auto_intensities_rate['nbr_observations'] = df_auto_intensities_rate[delta_years_str].notna().sum(axis=1)
df_auto_intensities_rate['number_sup_20'] = (df_auto_intensities_rate[delta_years_str] > 0.2).sum(axis=1)
df_auto_intensities_rate['number_inf_20'] = (df_auto_intensities_rate[delta_years_str] < -0.2).sum(axis=1)
df_auto_intensities_rate['number_sup_or_inf_20'] = ((df_auto_intensities_rate[delta_years_str] > 0.2)|(df_auto_intensities_rate[delta_years_str] < -0.2)).sum(axis=1)
# Number of observations / Number of rate sup to 20%
pd.crosstab(
    [df_auto_intensities_rate["scope"], df_auto_intensities_rate["nbr_observations"]],
    df_auto_intensities_rate["number_sup_or_inf_20"]
)

number_sup_or_inf_20     0
scope nbr_observations    
s3_d  0                 21
      1                  1
      2                  3
      3                  6
      4                  5

In [51]:
# df_auto_intensities_rate = df_auto_intensities_rate.set_index('isin')
df_ajd_3 = df_ajd_1.copy()
df_ajd_3 = df_ajd_3.set_index('isin')
df_ajd_3[delta_years_str] = (df_ajd_3[delta_years_str]
    .mask(((df_ajd_3[delta_years_str]>0.2)|(df_ajd_3[delta_years_str]<-0.2)) &
          (df_auto_intensities_rate[delta_years_str].notna()), 
    df_auto_intensities_rate[delta_years_str])
)

In [52]:
df_ajd_3['average_rate'] = df_ajd_3[delta_years_str].mean(axis=1)
df_ajd_3.groupby('scope')['average_rate'].describe()

,count,mean,std,min,25%,50%,75%,max
scope,,,,,,,,
s3_d,172.0,232.392077,2971.562139,-0.794042,0.469037,1.048598,2.147125,38976.849856


In [53]:
df_ajd_3['number_sup_20'] = (df_ajd_3[delta_years_str] > 0.2).sum(axis=1)
df_ajd_3['number_inf_20'] = (df_ajd_3[delta_years_str] < -0.2).sum(axis=1)
df_ajd_3['number_sup_or_inf_20'] = ((df_ajd_3[delta_years_str] > 0.2)|(df_ajd_3[delta_years_str] < -0.2)).sum(axis=1)
pd.DataFrame({
    "inf to -20%": [(df_ajd_3["number_inf_20"] > 0).sum()],
    "sup to 20%": [(df_ajd_3["number_sup_20"] > 0).sum()],
    "both": [(df_ajd_3["number_sup_or_inf_20"] > 0).sum()]
}, index=["Number of equities"]).reset_index()

,index,inf to -20%,sup to 20%,both
0,Number of equities,113,155,163


In [54]:
list_eq_auto = list(df[df['high_impact_sector']=='Automobiles']['isin'].unique())

In [55]:
df_intensities_prod = pd.read_excel("data/raw_data/moodys_raw/Mra_Rl_Temperature_Alignment_2050_2024_10.xlsx", sheet_name="Full")

# Create file for outliers treatment

In [86]:
first_year_available, last_year_available = 2019, 2024
delta_years = [f"{int(y)}-{y+1}" for y in range(first_year_available, last_year_available)]

In [93]:
import importlib
import outliers_treatment

importlib.reload(outliers_treatment)

from outliers_treatment import adj_1_keep_only_the_3_last_years, adj_2_unique_outlier_or_not_if_nbr_obs_inf_3, \
    adj_3_replace_emissions_with_intensities, dict_sectors_adjustements, treat_outlier_by_sector

In [94]:
df_adjusted = treat_outlier_by_sector(df, df_abs_em_rate, df_intensities_rate, ["Automobiles"],
                                      regions, dict_sectors_adjustements["Automobiles"],0.5, col_sector='high_impact_sector')

In [96]:
# df_adj_nbr_outliers_by_nbr_obs = 
pd.crosstab(
    [df_adjusted["nbr_observations"]], df_adjusted["number_all_outliers"]
)

number_all_outliers,0,1,2,3
nbr_observations,,,,
0,25,0,0,0
1,10,12,0,0
2,47,8,0,0
3,75,0,12,3


In [101]:
dict_thresholds = {}
for region in regions:
    dict_thresholds[region] = {}
    for sector in list_high_impact_sector:
        dict_thresholds[region][sector] = 0.5


In [102]:
dict_thresholds

{'Dev America US': {'Agriculture, forestry and fishing': 0.5,
  'Airlines': 0.5,
  'Aluminium': 0.5,
  'Automobiles': 0.5,
  'Banking': 0.5,
  'Cement': 0.5,
  'Chemicals': 0.5,
  'Coal mining': 0.5,
  'Consumer goods & services': 0.5,
  'Diversified mining': 0.5,
  'Electric utilities': 0.5,
  'Food producers': 0.5,
  'Industrials': 0.5,
  'Oil and gas': 0.5,
  'Paper': 0.5,
  'Real estate': 0.5,
  'Shipping': 0.5,
  'Steel': 0.5,
  'Transportation': 0.5,
  'non-HIMS': 0.5},
 'Emg Asia-Pac CN': {'Agriculture, forestry and fishing': 0.5,
  'Airlines': 0.5,
  'Aluminium': 0.5,
  'Automobiles': 0.5,
  'Banking': 0.5,
  'Cement': 0.5,
  'Chemicals': 0.5,
  'Coal mining': 0.5,
  'Consumer goods & services': 0.5,
  'Diversified mining': 0.5,
  'Electric utilities': 0.5,
  'Food producers': 0.5,
  'Industrials': 0.5,
  'Oil and gas': 0.5,
  'Paper': 0.5,
  'Real estate': 0.5,
  'Shipping': 0.5,
  'Steel': 0.5,
  'Transportation': 0.5,
  'non-HIMS': 0.5},
 'Emg Asia-Pac IN': {'Agriculture, fo

In [99]:
len(list_high_impact_sector)

20

In [103]:
list_high_impact_sector

['Agriculture, forestry and fishing',
 'Airlines',
 'Aluminium',
 'Automobiles',
 'Banking',
 'Cement',
 'Chemicals',
 'Coal mining',
 'Consumer goods & services',
 'Diversified mining',
 'Electric utilities',
 'Food producers',
 'Industrials',
 'Oil and gas',
 'Paper',
 'Real estate',
 'Shipping',
 'Steel',
 'Transportation',
 'non-HIMS']

In [100]:
len(regions)

12